# Minimal ReAct Agent with LangChain `create_agent` + Groq

In [ ]:
# %pip install -q langchain langchain-groq langchain-core langgraph

## 1.Access Model API key

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
GROQ_API_KEY  = os.getenv("GROQ_API_KEY")

## 2.Create react agent with llm, systemprompt, tools 

In [ ]:
from langchain.agents import create_agent          # replaces langgraph.prebuilt.create_react_agent
from langchain_groq import ChatGroq
from langchain_core.tools import tool

GROQ_MODEL = "qwen/qwen3-32b"

# --- Tools ---
@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location."""
    if location.lower() in ["sf", "san francisco"]:
        return "It's 60 degrees and foggy."
    return "It's 90 degrees and sunny."

@tool
def get_coolest_cities() -> str:
    """Get a list of the coolest cities."""
    return "nyc, sf"

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

tools = [get_weather, get_coolest_cities, multiply]

# --- Model ---
llm = ChatGroq(model=GROQ_MODEL)

# --- System Prompt ---
systemPrompt ="You are a helpful assistant. Use tools when needed to answer the user's question."

# --- Agent ---
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt = systempPrompt,
)

## 3.Run Agent (as a loop)

In [ ]:
# --- Run ---
user_question = "Will it rain tomorrow morning in Berlin?"

response = agent.invoke({
    "messages": [("user", user_question)]
})

# Final answer
print(response["messages"][-1].content)

In [ ]:
# Optional: inspect full message trace (tool calls + results)
for msg in response["messages"]:
    print(f"[{msg.__class__.__name__}] {msg.content}")